# Sprint 1 — BluaDiagnostics

## IA Conversacional para Check-up Digital

PoC acadêmica desenvolvida para o Challenge BluaDiagnostics utilizando:

- Prompt Engineering;
- memória conversacional;
- function calling simulado;
- integração com Ollama Cloud.

O notebook demonstra:

- system prompt clínico;
- guardrails;
- tools simuladas;
- fluxo de suporte ao beneficiário;
- contexto multi-turno.


## 1. Instalação das dependências

In [ ]:
!pip install -q ollama ipython

## 2. Configuração do Ollama Cloud

No menu lateral esquerdo do Colab:

🔑 Secrets

Crie:

```text
OLLAMA_API_KEY
```

Cole sua chave do Ollama Cloud.


In [ ]:
from google.colab import userdata
from ollama import Client

OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")

if not OLLAMA_API_KEY:
    raise ValueError("Configure o Secret OLLAMA_API_KEY no Colab.")

client = Client(
    host="https://ollama.com",
    headers={
        "Authorization": "Bearer " + OLLAMA_API_KEY
    }
)

MODEL_NAME = "gpt-oss:120b"

print("✅ Ollama Cloud configurado com sucesso")


## 3. Teste de conexão

In [ ]:
teste = client.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Responda apenas OK"
        }
    ],
    stream=False
)

print("Resposta do modelo:")
print(teste["message"]["content"])


## 4. System Prompt Clínico

O agente atua como assistente de apoio ao beneficiário da Care Plus.


In [ ]:
system_prompt = """
Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Respeite LGPD e minimize coleta de dados sensíveis.

ESCALADA HUMANA:
Encaminhe imediatamente situações de:
- falta de ar intensa;
- dor no peito;
- desmaio;
- confusão mental;
- sinais neurológicos;
- sangramentos importantes.

FORMATO:
1. Resumo do caso
2. Pontos de atenção
3. Próxima ação
4. Aviso médico
"""

print(system_prompt)


## 5. Dados simulados do paciente

In [ ]:
import json

paciente_mock = {
    "paciente_id": "PAC001",
    "nome": "Paciente Simulado",
    "idade": 42,
    "condicoes": ["hipertensão leve"],
    "medicamentos": ["losartana 50mg"],
    "alergias": ["dipirona"],
    "ultima_pressao": "135/85",
    "ultima_teleconsulta": "2026-05-10"
}

print(json.dumps(
    paciente_mock,
    indent=2,
    ensure_ascii=False
))


## 6. Function Calling Simulado

In [ ]:
def consultar_historico_paciente(paciente_id):
    print("🔧 Executando: consultar_historico_paciente")

    return paciente_mock


def verificar_interacoes_medicamentosas(medicamentos):
    print("🔧 Executando: verificar_interacoes_medicamentosas")

    if "ibuprofeno" in medicamentos:
        return {
            "risco": "moderado",
            "mensagem": "Possível atenção ao uso em paciente hipertenso."
        }

    return {
        "risco": "baixo",
        "mensagem": "Nenhuma interação relevante encontrada."
    }


def agendar_teleconsulta():
    print("🔧 Executando: agendar_teleconsulta")

    return {
        "status": "agendado",
        "data": "2026-05-20",
        "horario": "14:30"
    }

print("✅ Tools carregadas")


## 7. Memória Conversacional

In [ ]:
memoria = [
    {
        "role": "system",
        "content": system_prompt
    }
]

print("✅ Memória inicializada")


## 8. Função principal do agente

In [ ]:
from IPython.display import Markdown, display

def chamar_llm(mensagem_usuario):

    memoria.append({
        "role": "user",
        "content": mensagem_usuario
    })

    resposta = client.chat(
        model=MODEL_NAME,
        messages=memoria,
        options={
            "temperature": 0.3,
            "num_predict": 300
        },
        stream=False
    )

    texto = resposta["message"]["content"]

    memoria.append({
        "role": "assistant",
        "content": texto
    })

    display(Markdown(f"""
## 🤖 Resposta do BluaDiagnostics

{texto}
"""))

    return texto


## 9. Demonstração — Conversa Multi-turno

In [ ]:
chamar_llm(
    "Olá, sou hipertenso e hoje acordei com dor de cabeça leve."
)


In [ ]:
chamar_llm(
    "Minha pressão ficou 135 por 85 e senti tontura ao levantar."
)


## 10. Demonstração de memória

In [ ]:
print("📌 Histórico armazenado:\n")

for item in memoria:
    print(item["role"].upper())
    print(item["content"][:300])
    print("-" * 50)


## 11. Demonstração de Tool Calling

In [ ]:
historico = consultar_historico_paciente("PAC001")

print(json.dumps(
    historico,
    indent=2,
    ensure_ascii=False
))


In [ ]:
interacoes = verificar_interacoes_medicamentosas(
    ["losartana", "ibuprofeno"]
)

print(json.dumps(
    interacoes,
    indent=2,
    ensure_ascii=False
))


In [ ]:
teleconsulta = agendar_teleconsulta()

print(json.dumps(
    teleconsulta,
    indent=2,
    ensure_ascii=False
))


## 12. Resposta contextualizada usando tools

In [ ]:
contexto = f'''
Histórico do paciente:
{historico}

Interações medicamentosas:
{interacoes}

Teleconsulta:
{teleconsulta}

Gere uma orientação segura para o paciente.
'''

chamar_llm(contexto)


# Conclusão

A PoC demonstrou:

- uso de Prompt Engineering;
- system prompt clínico;
- memória conversacional;
- function calling simulado;
- integração com Ollama Cloud;
- guardrails clínicos;
- fluxo Human-in-the-Loop.

O projeto atende os requisitos principais da Sprint 1 do Challenge BluaDiagnostics.
